# HEBI explorer on a free Colab GPU

This notebook starts the HEBI web page inside Colab. You use the page as usual: upload `.h5ad` sections, run HEBI zero-shot or train first, and explore clusters, niches and genes. Nothing to code.

**Steps**
1. Menu **Runtime → Change runtime type → T4 GPU**, then **Save**.
2. Menu **Runtime → Run all**.
3. Wait for the last cell to print a link (about 5 minutes the first time), then click it.

Your files stay in this Colab session and are gone when it ends. Code: [github.com/roxana9/hebi](https://github.com/roxana9/hebi) (MIT). Weights: [huggingface.co/Roxana9/hebi](https://huggingface.co/Roxana9/hebi) (CC BY-NC 4.0, non-commercial research use).

In [ ]:
#@title 1. Install HEBI (about 3 to 5 minutes)
import subprocess, sys, torch
tv = torch.__version__.split("+")[0]
cu = "cu" + torch.version.cuda.replace(".", "") if torch.version.cuda else "cpu"
print("PyTorch", tv, cu, "| GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "none (set Runtime → T4 GPU)")
def pip(*args):
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", *args], check=True)
# torch_sparse must match the installed PyTorch; a prebuilt wheel is used when one exists
pip("torch-sparse", "-f", f"https://data.pyg.org/whl/torch-{tv}+{cu}.html")
pip("hebi @ git+https://github.com/roxana9/hebi@main", "fastapi", "uvicorn[standard]", "umap-learn", "huggingface_hub")
print("installed")

In [ ]:
#@title 2. Get the web page and the pretrained weights
import os, urllib.request
from huggingface_hub import hf_hub_download
WEB = "/content/hebi_web"; os.makedirs(WEB, exist_ok=True)
RAW = "https://raw.githubusercontent.com/roxana9/hebi/main/web"
for f in ["app.py", "index.html", "niche_merge.py"]:
    urllib.request.urlretrieve(f"{RAW}/{f}", f"{WEB}/{f}")
ckpt = hf_hub_download("Roxana9/hebi", "hebi_hybrid_oldL0.pt")
vocab = hf_hub_download("Roxana9/hebi", "default_gene_vocab.json")
print("ready:", ckpt)

In [ ]:
#@title 3. Start HEBI and open the page
import subprocess, time, json, urllib.request, os
env = dict(os.environ, HEBI_CHECKPOINT=ckpt, HEBI_VOCAB=vocab, HEBI_WEB_HOST="127.0.0.1", HEBI_WEB_PORT="8000",
           HEBI_WEB_WORKDIR="/content/hebi_jobs", HEBI_WEB_MAX_UPLOAD_GB="4", HEBI_WEB_MAX_EPOCHS="200")
log = open("/content/hebi_web.log", "w")
server = subprocess.Popen([sys.executable, "app.py"], cwd=WEB, env=env, stdout=log, stderr=subprocess.STDOUT)
print("starting HEBI (loads PyTorch and the model, up to a few minutes)", end="")
for _ in range(120):
    time.sleep(5); print(".", end="")
    try:
        if json.load(urllib.request.urlopen("http://127.0.0.1:8000/api/health", timeout=5)).get("ok"): break
    except Exception: pass
    if server.poll() is not None: break
print()
if server.poll() is not None:
    print(open("/content/hebi_web.log").read()[-3000:])
else:
    from google.colab import output
    print("HEBI is running. Click the link below; keep this tab open while you work.")
    output.serve_kernel_port_as_window(8000, path="/")

If something goes wrong, run the cell below and send the last lines of the log.

In [ ]:
print(open("/content/hebi_web.log").read()[-4000:])